In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier


In [2]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [3]:
import joblib

sns.set_style("whitegrid")

print("Libraries loaded successfully")


Libraries loaded successfully


Step 3 — Create the dataset
Because 2 million rows can be unnecessarily heavy for a student laptop, I recommend initially creating 100,000 tickets for development/testing.

Your final report can state that the architecture is designed to scale to the stated 2 million annual tickets / 50,000 events-per-second streaming requirement.


In [4]:
np.random.seed(42)

N = 100_000

issue_categories = [
    "Network",
    "Software",
    "Hardware",
    "Access",
    "Security",
    "Email",
    "Database",
    "VPN"
]

priorities = [
    "Low",
    "Medium",
    "High",
    "Critical"
]

client_types = [
    "Small",
    "Medium",
    "Enterprise"
]

df = pd.DataFrame({
    "ticket_id": np.arange(1, N + 1),

    "issue_category": np.random.choice(
        issue_categories,
        N,
        p=[0.18, 0.20, 0.15, 0.15, 0.08, 0.10, 0.06, 0.08]
    ),

    "priority": np.random.choice(
        priorities,
        N,
        p=[0.30, 0.40, 0.23, 0.07]
    ),

    "client_type": np.random.choice(
        client_types,
        N,
        p=[0.50, 0.30, 0.20]
    )
})

df.head()

,ticket_id,issue_category,priority,client_type
0,1,Software,Medium,Small
1,2,VPN,Medium,Small
2,3,Security,Medium,Small
3,4,Access,Medium,Small
4,5,Network,Medium,Small


Step 4 — Generate resolution time
Add another Code cell

In [5]:
base_resolution = {
    "Network": 4.5,
    "Software": 3.0,
    "Hardware": 5.0,
    "Access": 1.5,
    "Security": 6.0,
    "Email": 2.0,
    "Database": 5.5,
    "VPN": 3.5
}

priority_multiplier = {
    "Low": 0.8,
    "Medium": 1.0,
    "High": 1.3,
    "Critical": 1.7
}

df["resolution_time"] = df.apply(
    lambda row:
    base_resolution[row["issue_category"]]
    * priority_multiplier[row["priority"]]
    * np.random.lognormal(mean=0, sigma=0.25),
    axis=1
)

df["resolution_time"] = df["resolution_time"].round(2)

df.head()

,ticket_id,issue_category,priority,client_type,resolution_time
0,1,Software,Medium,Small,2.94
1,2,VPN,Medium,Small,2.64
2,3,Security,Medium,Small,5.50
3,4,Access,Medium,Small,1.79
4,5,Network,Medium,Small,3.98


Step 5 — Generate engineer hourly cost

In [6]:
engineer_cost_by_category = {
    "Network": 55,
    "Software": 45,
    "Hardware": 50,
    "Access": 35,
    "Security": 75,
    "Email": 40,
    "Database": 70,
    "VPN": 50
}

df["cost_per_engineer_hour"] = (
    df["issue_category"]
    .map(engineer_cost_by_category)
)

df["cost_per_engineer_hour"] = (
    df["cost_per_engineer_hour"]
    * np.random.uniform(0.90, 1.10, N)
).round(2)


Step 6 — Generate escalation count
We'll make escalations more likely for difficult/high-priority tickets

In [7]:
priority_escalation_probability = {
    "Low": 0.05,
    "Medium": 0.10,
    "High": 0.18,
    "Critical": 0.28
}

category_escalation_probability = {
    "Network": 0.15,
    "Software": 0.10,
    "Hardware": 0.18,
    "Access": 0.05,
    "Security": 0.25,
    "Email": 0.07,
    "Database": 0.22,
    "VPN": 0.14
}

def generate_escalation(row):
    probability = (
        priority_escalation_probability[row["priority"]]
        + category_escalation_probability[row["issue_category"]]
    ) / 2

    return np.random.binomial(3, probability)

df["escalation_count"] = df.apply(
    generate_escalation,
    axis=1
)

df["escalation_count"].value_counts().sort_index()


,count
escalation_count,
0,67868
1,27282
2,4567
3,283


Step 7 — Generate customer satisfaction
Now we need CSAT because management wants

In [10]:


df["csat"] = (
    4.7
    - (df["resolution_time"] * 0.06)
    - (df["escalation_count"] * 0.20)
    + np.random.normal(0, 0.25, N)
)

df["csat"] = df["csat"].clip(1, 5).round(2)

df["csat"].describe()


,csat
count,100000.000000
mean,4.382127
std,0.312016
min,2.770000
25%,4.180000
50%,4.400000
75%,4.600000
max,5.000000


Step 8 — Check your dataset
Now run:

In [13]:
df.head()


df.shape

(100000, 8)


df.columns.tolist()

[
'ticket_id',
'issue_category',
'priority',
'client_type',
'resolution_time',
'cost_per_engineer_hour',
'escalation_count',
'csat'
]

['ticket_id',
 'issue_category',
 'priority',
 'client_type',
 'resolution_time',
 'cost_per_engineer_hour',
 'escalation_count',
 'csat']

Step 9 — Add missing data
Your problem specifically says:

Missing data: up to 20% randomly missing

So let's intentionally introduce missing values to test our solution.

For example, we'll create approximately 10% missing values in selected fields:


In [14]:
missing_columns = [
    "issue_category",
    "priority",
    "resolution_time",
    "cost_per_engineer_hour",
    "client_type",
    "escalation_count"
]

for col in missing_columns:
    mask = np.random.random(N) < 0.10
    df.loc[mask, col] = np.nan

In [15]:
df.isnull().sum()

,0
ticket_id,0
issue_category,9922
priority,10082
client_type,9979
resolution_time,10013
cost_per_engineer_hour,9966
escalation_count,9959
csat,0


In [16]:
(df.isnull().sum() / len(df) * 100).round(2)

,0
ticket_id,0.00
issue_category,9.92
priority,10.08
client_type,9.98
resolution_time,10.01
cost_per_engineer_hour,9.97
escalation_count,9.96
csat,0.00


Step 10 — Save your dataset
Now you actually have a CSV

In [17]:
df.to_csv(
    "IT_support_dataset.csv",
    index=False
)

print("Dataset saved successfully!")

Dataset saved successfully!


Step 11 — Now start the actual analysis
Before doing ML, clean the data

In [18]:
df["issue_category"] = df["issue_category"].fillna("Unknown")

df["priority"] = df["priority"].fillna("Unknown")

df["client_type"] = df["client_type"].fillna("Unknown")

df["resolution_time"] = df["resolution_time"].fillna(
    df["resolution_time"].median()
)

df["cost_per_engineer_hour"] = df[
    "cost_per_engineer_hour"
].fillna(
    df["cost_per_engineer_hour"].median()
)

df["escalation_count"] = df[
    "escalation_count"
].fillna(0)

For CSAT

In [19]:
df["csat"] = df["csat"].fillna(
    df["csat"].median()
)

Step 12 — Calculate ticket cost

In [20]:
df["ticket_cost"] = (
    df["resolution_time"]
    * df["cost_per_engineer_hour"]
)

print(
    "Total Support Cost:",
    df["ticket_cost"].sum()
)

print(
    "Average Ticket Cost:",
    df["ticket_cost"].mean()
)

Total Support Cost: 21045861.456300005
Average Ticket Cost: 210.45861456300005


Step 13 — Calculate escalation rate

In [21]:
df["escalated"] = (
    df["escalation_count"] > 0
).astype(int)

escalation_rate = (
    df["escalated"].mean() * 100
)

print(
    "Escalation Rate:",
    round(escalation_rate, 2),
    "%"
)

Escalation Rate: 28.87 %


Step 14 — Calculate CSAT

In [22]:
average_csat = df["csat"].mean()

print(
    "Average CSAT:",
    round(average_csat, 2)
)

print("Target CSAT: > 4.2")

Average CSAT: 4.38
Target CSAT: > 4.2


Step 15 — Calculate the 20% cost target

In [23]:
current_cost = df["ticket_cost"].sum()

required_savings = current_cost * 0.20

target_cost = current_cost * 0.80

print("Current Cost:", current_cost)
print("Required 20% Savings:", required_savings)
print("Target Cost:", target_cost)

Current Cost: 21045861.456300005
Required 20% Savings: 4209172.291260001
Target Cost: 16836689.165040005
